# Problem 55: RAG Query Rewriting

**Difficulty:** Medium | **Framework:** LangChain

**Categories:** rag, prompt-design

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app/problems/rag-query-rewriting).

## 1. Install Dependencies

In [ ]:
!pip install -q langchain langchain-openai langchain-community langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The user's original query must be rewritten before retrieval
- The rewritten query must improve retrieval relevance
- The original user intent must be preserved in the rewrite
- The rewriting step must use an LLM to reformulate the query


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings()

documents = [
    Document(page_content="Authentication uses OAuth 2.0 with JWT tokens. Tokens expire after 24 hours."),
    Document(page_content="Rate limiting: API allows 1000 requests per minute per API key."),
    Document(page_content="Error handling: 4xx errors return JSON with 'error' and 'message' fields."),
    Document(page_content="Pagination: Use 'cursor' parameter for paginated endpoints. Max page size is 100."),
    Document(page_content="Webhook events are signed with HMAC-SHA256. Verify the X-Signature header."),
]

vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based on context.\n\nContext: {context}"),
    ("human", "{question}"),
])

# BUG: User's vague query is passed directly to retriever with no rewriting
def ask(question: str) -> str:
    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])
    chain = prompt | llm
    result = chain.invoke({"context": context, "question": question})
    return result.content

# Vague query — retriever may not find the right docs
print(ask("How do I log in?"))

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Add a query rewriting step before retrieval that uses the LLM to generate a better search query
2. The rewritten query should be more specific and use terminology likely to appear in the documents
3. Consider generating multiple query variations and retrieving for each to improve recall


## 7. Evaluation Criteria

Check your solution against these criteria:

- Rewrites vague queries
- Improves retrieval relevance
- Preserves user intent
